# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
This dataset is accessible via a [FAIR² Croissant](https://mlcommons.org/croissant/) metadata schema, provided through the following URL:


In [ ]:
# Install mlcroissant if not present
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset (Croissant metadata)
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
if hasattr(dataset.metadata, 'keywords'):
    print(f"Keywords: {dataset.metadata.keywords}")
print(f"License: {dataset.metadata.license}")


## 2. Data Overview
Review the available record sets, fields, and column `@id`s in the dataset.

In [ ]:
# List available record sets and their fields (@id)
print("Available record sets and their fields:")
for rs in dataset.record_sets:
    # The record set's @id and name
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    # List fields/columns and their @ids
    print("  Fields (columns):")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, data type: {field.data_type}")
    print()
if not dataset.record_sets:
    print("No explicit RecordSet (cr:RecordSet) listed; trying to access inferred record sets via `dataset.records()`.\n")
    # Try to access all the records (may be one record set)
    # Use the most common pattern: If only one CSV or file, its @id is used
    # Show available distributions
    if hasattr(dataset.metadata, 'distribution'):
        for dist in dataset.metadata.distribution:
            print(f"Distribution @id: {dist['@id']}")


## 3. Data Extraction
Load data from record set(s) into a DataFrame for analysis. All Croissant entities are referenced by their `@id` for clarity and reproducibility.

In [ ]:
# In this schema, explicit record sets are not listed in dataset.metadata.recordSet or dataset.record_sets.
# However, Croissant allows using the DataFileObject or distribution as sources (see the previous cell for distribution ids).
# For demonstration, we'll use the detected distribution @id as the record set.
#
# If record set @ids were defined, you'd use those. We proceed by inferring from the distribution.

record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # From distribution @id above

# Use the mlcroissant records() interface with this id
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)

# Show columns
print(f"Columns found in {record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps like filtering, normalization, and grouping. For demonstration, we select a numeric protein-related field if it exists in the columns (e.g., `MW` for molecular weight, which is typical in proteomics). All columns and references are by `@id` (column names correspond to their Croissant field `@id`).

In [ ]:
# First, list available columns
print(f"Available fields (@id/column names): {df.columns.tolist()}")

# Choose a numeric field for filtering and normalization. We'll pick 'MW' (Molecular Weight) if present. Otherwise, pick another numeric field.
candidate_numeric_fields = ['MW', 'Coverage', 'iBAQ', 'Score', 'Abundance']
numeric_field = None
for f in candidate_numeric_fields:
    if f in df.columns:
        numeric_field = f
        break
if numeric_field is None:
    print("No recognized numeric field found for demonstration.")
    numeric_field = df.select_dtypes('number').columns[0] if len(df.select_dtypes('number').columns) else df.columns[0]

print(f"Using numeric field: {numeric_field}")

threshold = 25000  # For demonstration, filter on MW > 25000 if MW is present

filtered_df = df.copy()
if numeric_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records: {filtered_df.shape[0]} with {numeric_field} > {threshold}")
else:
    print(f"Unable to filter; {numeric_field} is not numeric or missing.")

# Normalization (z-score)
if numeric_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

# Select a grouping field by unique protein/Accession or sample/column if present
candidate_group_fields = ['Accession', 'Description', 'Sample']
group_field = None
for f in candidate_group_fields:
    if f in df.columns:
        group_field = f
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped mean by {group_field} (showing first 5 rows):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or field relationships. The following example produces a histogram of molecular weight and a scatter plot of MW vs. another quantitative field if present.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram of the numeric field
if numeric_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

# Scatter plot MW vs. iBAQ or Abundance or Score if present
for field2 in ['iBAQ', 'Abundance', 'Score', 'Coverage']:
    if field2 in df.columns and field2 != numeric_field and pd.api.types.is_numeric_dtype(df[field2]):
        plt.scatter(df[numeric_field], df[field2], alpha=0.6)
        plt.xlabel(numeric_field)
        plt.ylabel(field2)
        plt.title(f'{numeric_field} vs. {field2}')
        plt.show()
        break

## 6. Conclusion

- Successfully loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant` referencing all entities by their schema `@id`.
- Fields and columns are accessible by their Croissant-defined names and can be manipulated in pandas DataFrames.
- Initial analysis focused on protein molecular weight and abundance, but Croissant metadata allows for highly reproducible workflows for any field or record set.
- For advanced analysis, consult the schema's field definitions and the official Croissant and `mlcroissant` documentation.